In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

import xgboost as xgb
import optuna

from tqdm import tqdm

In [2]:
train_df = pd.read_csv("training(2003-2023).csv")
test_df = pd.read_csv("test(2024-25).csv")

print(train_df.shape)
print(test_df.shape)

(20988361, 14)
(1182188, 14)


In [3]:
FEATURES = [
    "CHLOR_A",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "LAT_scaled",
    "LON_scaled"
]

TARGET = "PHYTOBLOOM"

In [4]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [31]:
def objective(trial):

    # params = {

    #     "objective": "multi:softprob",
    #     "num_class": 3,

    #     "n_estimators":
    #         trial.suggest_int(
    #             "n_estimators",
    #             100,
    #             500,
    #             step=50
    #         ),

    #     "max_depth":
    #         trial.suggest_int(
    #             "max_depth",
    #             4,
    #             12
    #         ),

    #     "learning_rate":
    #         trial.suggest_float(
    #             "learning_rate",
    #             0.01,
    #             0.3,
    #             log=True
    #         ),

    #     "subsample":
    #         trial.suggest_float(
    #             "subsample",
    #             0.6,
    #             1.0
    #         ),

    #     "colsample_bytree":
    #         trial.suggest_float(
    #             "colsample_bytree",
    #             0.6,
    #             1.0
    #         ),

    #     "min_child_weight":
    #         trial.suggest_int(
    #             "min_child_weight",
    #             1,
    #             20
    #         ),

    #     "gamma":
    #         trial.suggest_float(
    #             "gamma",
    #             1e-5,
    #             1.0,
    #             log=True
    #         ),

    #     "lambda":
    #         trial.suggest_float(
    #             "lambda",
    #             1e-3,
    #             10,
    #             log=True
    #         ),

    #     "alpha":
    #         trial.suggest_float(
    #             "alpha",
    #             1e-3,
    #             5,
    #             log=True
    #         ),

    #     "grow_policy":
    #         trial.suggest_categorical(
    #             "grow_policy",
    #             ["depthwise", "lossguide"]
    #         ),

    #     "tree_method": "hist",

    #     "random_state": 42,

    #     "n_jobs": -1
    # }

    # params = {

    #     "n_estimators":
    #     trial.suggest_int(
    #         "n_estimators",
    #         30,
    #         80,
    #         step=10
    #     ),

    #     "max_depth":
    #     trial.suggest_int(
    #         "max_depth",
    #         1,
    #         6
    #     ),

    #     "learning_rate":
    #     trial.suggest_float(
    #         "learning_rate",
    #         0.03,
    #         0.10
    #     ),

    #     "min_child_weight":
    #     trial.suggest_int(
    #         "min_child_weight",
    #         10,
    #         18
    #     ),

    #     "subsample":
    #     trial.suggest_float(
    #         "subsample",
    #         0.50,
    #         0.85
    #     ),

    #     "colsample_bytree":
    #     trial.suggest_float(
    #         "colsample_bytree",
    #         0.65,
    #         0.80
    #     ),

    #     "gamma":
    #     trial.suggest_float(
    #         "gamma",
    #         0.0,
    #         0.05
    #     ),

    #     "lambda":
    #     trial.suggest_float(
    #         "lambda",
    #         0.001,
    #         0.01,
    #         log=True
    #     ),

    #     "alpha":
    #     trial.suggest_float(
    #         "alpha",
    #         0.001,
    #         0.02,
    #         log=True
    #     )
    # }

    params = {

        "n_estimators":
        trial.suggest_int(
            "n_estimators",
            10,
            70,
            step=10
        ),

        "max_depth":
        trial.suggest_int(
            "max_depth",
            2,
            8
        ),

        "learning_rate":
        trial.suggest_float(
            "learning_rate",
            0.03,
            0.15
        ),

        "min_child_weight":
        trial.suggest_int(
            "min_child_weight",
            10,
            18
        ),

        "subsample":
        trial.suggest_float(
            "subsample",
            0.50,
            0.85
        ),

        "colsample_bytree":
        trial.suggest_float(
            "colsample_bytree",
            0.65,
            0.80
        ),

        "gamma":
        trial.suggest_float(
            "gamma",
            0.0,
            0.05
        ),

        "lambda":
        trial.suggest_float(
            "lambda",
            0.001,
            0.01,
            log=True
        ),

        "alpha":
        trial.suggest_float(
            "alpha",
            0.001,
            0.02,
            log=True
        )
    }

    skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

    scores = []

    for train_idx, val_idx in skf.split(X_train,y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = xgb.XGBClassifier(
            **params
        )

        model.fit(
            X_tr,
            y_tr,
            verbose=False
        )

        pred = model.predict(X_val)

        macro_f1 = f1_score(
            y_val,
            pred,
            average="macro"
        )

        scores.append(
            macro_f1
        )

    return np.mean(scores)

In [13]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-06-24 12:09:18,872] A new study created in memory with name: no-name-7908a2dc-fe5b-4a48-9e58-c05baf73a499


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-06-24 12:33:59,945] Trial 0 finished with value: 0.9347852268967994 and parameters: {'n_estimators': 150, 'max_depth': 5, 'learning_rate': 0.24185210764823528, 'subsample': 0.6460050564514498, 'colsample_bytree': 0.6597355908371025, 'min_child_weight': 13, 'gamma': 1.0457661097563821e-05, 'lambda': 1.844295039318334, 'alpha': 1.4830214178222523, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 0.9347852268967994.
[I 2026-06-24 14:09:40,257] Trial 1 finished with value: 0.9385135108679756 and parameters: {'n_estimators': 500, 'max_depth': 12, 'learning_rate': 0.016281695396146115, 'subsample': 0.7507463273900259, 'colsample_bytree': 0.83282516039172, 'min_child_weight': 18, 'gamma': 0.004473123047487134, 'lambda': 0.10147397202253616, 'alpha': 0.001058089429099311, 'grow_policy': 'depthwise'}. Best is trial 1 with value: 0.9385135108679756.
[I 2026-06-24 15:17:10,479] Trial 2 finished with value: 0.9351980970250476 and parameters: {'n_estimators': 350, 'max_depth': 10, '

In [15]:
print("Best Macro F1:")
print(study.best_value)

print("\nBest Parameters:")
study.best_params

Best Macro F1:
0.9419934990253955

Best Parameters:


{'n_estimators': 400,
 'max_depth': 11,
 'learning_rate': 0.160150368880871,
 'subsample': 0.6798136276263954,
 'colsample_bytree': 0.7243238118279334,
 'min_child_weight': 14,
 'gamma': 0.014344298946540327,
 'lambda': 0.0027281560811716794,
 'alpha': 0.007634436209519525,
 'grow_policy': 'lossguide'}

In [18]:
study2 = optuna.create_study(direction="maximize")

study2.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-06-27 03:22:15,259] A new study created in memory with name: no-name-b8207c98-981f-484a-9cd5-a090f5b84dd0


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-06-27 03:34:13,691] Trial 0 finished with value: 0.9121968687854963 and parameters: {'n_estimators': 40, 'max_depth': 2, 'learning_rate': 0.047914439391425825, 'min_child_weight': 11, 'subsample': 0.6267326399623052, 'colsample_bytree': 0.6981529849777277, 'gamma': 0.02205392010037094, 'lambda': 0.0012070757607308567, 'alpha': 0.011698902022926278}. Best is trial 0 with value: 0.9121968687854963.
[I 2026-06-27 04:00:09,233] Trial 1 finished with value: 0.9301283120526602 and parameters: {'n_estimators': 80, 'max_depth': 2, 'learning_rate': 0.08883302285030262, 'min_child_weight': 16, 'subsample': 0.5653541875182522, 'colsample_bytree': 0.7660284603044558, 'gamma': 0.0330318336100936, 'lambda': 0.00921567332126062, 'alpha': 0.005312820857270492}. Best is trial 1 with value: 0.9301283120526602.
[I 2026-06-27 04:35:36,503] Trial 2 finished with value: 0.9288983367403685 and parameters: {'n_estimators': 40, 'max_depth': 3, 'learning_rate': 0.07508371939364725, 'min_child_weight': 1

In [19]:
print("Best Macro F1:")
print(study2.best_value)

print("\nBest Parameters:")
study2.best_params

Best Macro F1:
0.9317767451508058

Best Parameters:


{'n_estimators': 60,
 'max_depth': 4,
 'learning_rate': 0.09978551341356359,
 'min_child_weight': 13,
 'subsample': 0.6770920535070198,
 'colsample_bytree': 0.723049278515438,
 'gamma': 0.014891898198791116,
 'lambda': 0.005005852939878152,
 'alpha': 0.0010681845124340952}

In [32]:
study3 = optuna.create_study(direction="maximize")

study3.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-06-28 16:32:15,298] A new study created in memory with name: no-name-896ddb9c-ea71-483d-991a-56c97edc50bd


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-06-28 16:41:15,849] Trial 0 finished with value: 0.8866716985090222 and parameters: {'n_estimators': 40, 'max_depth': 5, 'learning_rate': 0.030405314361087867, 'min_child_weight': 16, 'subsample': 0.5396316067157367, 'colsample_bytree': 0.6757683575494218, 'gamma': 0.042499474760294656, 'lambda': 0.001289716648284723, 'alpha': 0.009448301684613816}. Best is trial 0 with value: 0.8866716985090222.
[I 2026-06-28 16:48:26,914] Trial 1 finished with value: 0.9299322640086267 and parameters: {'n_estimators': 30, 'max_depth': 5, 'learning_rate': 0.10103149519660198, 'min_child_weight': 12, 'subsample': 0.6396430875918652, 'colsample_bytree': 0.7092576907327343, 'gamma': 0.014174990998401632, 'lambda': 0.003182963845120234, 'alpha': 0.003024857292449281}. Best is trial 1 with value: 0.9299322640086267.
[I 2026-06-28 16:54:05,077] Trial 2 finished with value: 0.9214836517692181 and parameters: {'n_estimators': 20, 'max_depth': 7, 'learning_rate': 0.08300937685438435, 'min_child_weight'

In [16]:
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [17]:
optuna.visualization.plot_parallel_coordinate(study)

In [18]:
optuna.visualization.plot_slice(study)

In [19]:
optuna.visualization.plot_param_importances(study)

In [33]:
best_params = study3.best_params
best_params

{'n_estimators': 70,
 'max_depth': 8,
 'learning_rate': 0.1349325608942457,
 'min_child_weight': 14,
 'subsample': 0.5961850967666071,
 'colsample_bytree': 0.742279400803029,
 'gamma': 0.03472355046243029,
 'lambda': 0.0074299131797073556,
 'alpha': 0.0049279110449704224}

In [8]:
best_params ={'n_estimators': 60,
 'max_depth': 4,
 'learning_rate': 0.09978551341356359,
 'min_child_weight': 13,
 'subsample': 0.6770920535070198,
 'colsample_bytree': 0.723049278515438,
 'gamma': 0.014891898198791116,
 'lambda': 0.005005852939878152,
 'alpha': 0.0010681845124340952
}

In [8]:

import joblib

final_model =  joblib.load('lgb_model_NAS.pkl')

final_model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.359926 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 916
[LightGBM] [Info] Number of data points in the train set: 20988361, number of used features: 7
[LightGBM] [Info] Start training from score -0.227400
[LightGBM] [Info] Start training from score -2.977051
[LightGBM] [Info] Start training from score -1.880887
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,num_leaves,66
,max_depth,4
,learning_rate,0.05100951282436649
,n_estimators,30
,objective,'multiclass'
,min_split_gain,0.006075023874388366
,min_child_samples,34
,subsample,0.897944887767238
,colsample_bytree,0.7551178667909587
,reg_alpha,0.12087874480114016
,reg_lambda,0.11024868381878898


In [9]:
y_pred1 = final_model.predict(
    X_train
)
y_pred = final_model.predict(
    X_test
)

In [10]:
print("TRAIN RESULTS")
print(
    classification_report(
        y_train,
        y_pred1
    )
)

macro_f1 = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

print(
    "Train Macro F1:",
    macro_f1
)
print()
print("TEST RESULTS")
print(
    classification_report(
        y_test,
        y_pred
    )
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(
    "Test Macro F1:",
    macro_f1
)

TRAIN RESULTS
              precision    recall  f1-score   support

           0       1.00      1.00      1.00  16719377
           1       0.83      0.87      0.85   1069207
           2       0.94      0.94      0.94   3199777

    accuracy                           0.98  20988361
   macro avg       0.92      0.94      0.93  20988361
weighted avg       0.98      0.98      0.98  20988361

Train Macro F1: 0.9296942668815399

TEST RESULTS
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    956898
           1       0.81      0.89      0.85     75876
           2       0.93      0.88      0.91    149414

    accuracy                           0.98   1182188
   macro avg       0.91      0.92      0.92   1182188
weighted avg       0.98      0.98      0.98   1182188

Test Macro F1: 0.9176816948829142


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    956898
           1       0.80      0.89      0.84     75876
           2       0.93      0.88      0.91    149414

    accuracy                           0.98   1182188
   macro avg       0.91      0.92      0.92   1182188
weighted avg       0.98      0.98      0.98   1182188

Test Macro F1: 0.915229630211178


In [26]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[954809    140   1949]
 [   196  65699   9981]
 [   898  16878 131638]]


In [5]:
best_params={'n_estimators': 400,
 'max_depth': 11,
 'learning_rate': 0.160150368880871,
 'subsample': 0.6798136276263954,
 'colsample_bytree': 0.7243238118279334,
 'min_child_weight': 14,
 'gamma': 0.014344298946540327,
 'lambda': 0.0027281560811716794,
 'alpha': 0.007634436209519525,
 'grow_policy': 'lossguide'}


In [2]:
from sklearn.metrics import f1_score
import copy
import xgboost as xgb
import pandas as pd

def tune_one_parameter(param_name, values):

    results = []

    for value in values:

        params = copy.deepcopy(best_params)
        params[param_name] = value

        model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            **params
        )

        model.fit(X_train, y_train)

        # Train prediction
        train_pred = model.predict(X_train)
        train_f1 = f1_score(
            y_train,
            train_pred,
            average="macro"
        )

        # Test prediction
        test_pred = model.predict(X_test)
        test_f1 = f1_score(
            y_test,
            test_pred,
            average="macro"
        )

        gap = train_f1 - test_f1

        results.append([value, train_f1, test_f1, gap])

        print(
            f"{param_name}={value} | "
            f"Train={train_f1:.4f} | "
            f"Test={test_f1:.4f} | "
            f"Gap={gap:.4f}"
        )

In [32]:
depth_results = tune_one_parameter(
    "max_depth",
    [8,9,10,11,12,13,14]
)

depth_results

max_depth=8 | Train=0.9415 | Test=0.9109 | Gap=0.0306
max_depth=9 | Train=0.9432 | Test=0.9101 | Gap=0.0331
max_depth=10 | Train=0.9454 | Test=0.9095 | Gap=0.0360
max_depth=11 | Train=0.9477 | Test=0.9085 | Gap=0.0392
max_depth=12 | Train=0.9500 | Test=0.9077 | Gap=0.0423
max_depth=13 | Train=0.9527 | Test=0.9072 | Gap=0.0455
max_depth=14 | Train=0.9555 | Test=0.9062 | Gap=0.0493


,max_depth,Train Macro F1,Test Macro F1,Generalization Gap
0,8,0.941470,0.910851,0.030619
1,9,0.943193,0.910073,0.033119
2,10,0.945434,0.909481,0.035953
3,11,0.947651,0.908489,0.039162
4,12,0.950034,0.907696,0.042339
5,13,0.952689,0.907198,0.045491
6,14,0.955481,0.906205,0.049276


In [33]:
tune_one_parameter(
    "max_depth",
    [4,5,6,7]
)

max_depth=4 | Train=0.9352 | Test=0.9150 | Gap=0.0202
max_depth=5 | Train=0.9367 | Test=0.9136 | Gap=0.0231
max_depth=6 | Train=0.9381 | Test=0.9130 | Gap=0.0251
max_depth=7 | Train=0.9396 | Test=0.9119 | Gap=0.0277


,max_depth,Train Macro F1,Test Macro F1,Generalization Gap
0,4,0.935221,0.915034,0.020186
1,5,0.936681,0.913564,0.023117
2,6,0.938146,0.913019,0.025127
3,7,0.939635,0.911902,0.027732


In [ ]:
tune_one_parameter(
    "min_child_weight",
    [12,13,15,16]
)

max_depth=1 | Train=0.9307 | Test=0.9176 | Gap=0.0131
max_depth=2 | Train=0.9327 | Test=0.9164 | Gap=0.0163
max_depth=3 | Train=0.9339 | Test=0.9156 | Gap=0.0183


,max_depth,Train Macro F1,Test Macro F1,Generalization Gap
0,1,0.930686,0.917569,0.013117
1,2,0.932685,0.916365,0.016319
2,3,0.933873,0.915610,0.018262


In [35]:
tune_one_parameter(
    "min_child_weight",
    [12,13,15,16]
)

min_child_weight=12 | Train=0.9479 | Test=0.9085 | Gap=0.0394
min_child_weight=13 | Train=0.9479 | Test=0.9085 | Gap=0.0393
min_child_weight=15 | Train=0.9474 | Test=0.9086 | Gap=0.0388
min_child_weight=16 | Train=0.9474 | Test=0.9084 | Gap=0.0389


,min_child_weight,Train Macro F1,Test Macro F1,Generalization Gap
0,12,0.947920,0.908512,0.039408
1,13,0.947855,0.908523,0.039332
2,15,0.947421,0.908636,0.038785
3,16,0.947361,0.908419,0.038942


In [11]:
tune_one_parameter(
    "n_estimators",
    [350,420,470]
)

n_estimators=350 | Train=0.9468 | Test=0.9088 | Gap=0.0380
n_estimators=420 | Train=0.9480 | Test=0.9082 | Gap=0.0397


KeyboardInterrupt: 

In [12]:
tune_one_parameter(
    "n_estimators",
    [200,250,300]
)

n_estimators=200 | Train=0.9439 | Test=0.9100 | Gap=0.0340
n_estimators=250 | Train=0.9450 | Test=0.9096 | Gap=0.0354
n_estimators=300 | Train=0.9461 | Test=0.9090 | Gap=0.0370


In [13]:
tune_one_parameter(
    "n_estimators",
    [50,100,150]
)

n_estimators=50 | Train=0.9386 | Test=0.9137 | Gap=0.0249
n_estimators=100 | Train=0.9409 | Test=0.9119 | Gap=0.0291
n_estimators=150 | Train=0.9425 | Test=0.9109 | Gap=0.0316


In [14]:
tune_one_parameter(
    "subsample",
    [0.51,0.73,0.82]
)

subsample=0.51 | Train=0.9469 | Test=0.9086 | Gap=0.0383
subsample=0.73 | Train=0.9477 | Test=0.9083 | Gap=0.0393
subsample=0.82 | Train=0.9477 | Test=0.9087 | Gap=0.0390


In [16]:
tune_one_parameter(
    "lambda",
    [0.0011,0.0021,0.0034]
)

lambda=0.0011 | Train=0.9476 | Test=0.9085 | Gap=0.0391
lambda=0.0021 | Train=0.9477 | Test=0.9085 | Gap=0.0392
lambda=0.0034 | Train=0.9476 | Test=0.9085 | Gap=0.0392
